# Vehicle Detection using YOLOv11
This notebook implements vehicle detection using YOLOv11 model in Google Colab.

## Setup and Installation

In [ ]:
# Install required packages
!pip install ultralytics opencv-python-headless torch

In [ ]:
# Import necessary libraries
import cv2
from ultralytics import YOLO
import torch
import gc
from google.colab import files

## Configure GPU and Load Model

In [ ]:
# Check for CUDA availability and set device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('='*50)
if DEVICE == 'cuda':
    torch.backends.cudnn.benchmark = True
    print(f"RUNNING ON GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("RUNNING ON CPU")
    print("GPU is not available")
print('='*50)

# Load YOLOv11 model
model = YOLO('yolo11x.pt')
model.to(DEVICE)
model.verbose = False

## Define Colors and Detection Function

In [ ]:
# Define colors for different labels
label_colors = {
    'car': (0, 255, 0),        # Green
    'truck': (255, 165, 0),    # Orange
    'motorcycle': (0, 0, 255),  # Red
    'bus': (255, 255, 0),      # Cyan
    'van': (255, 0, 0)         # Blue
}

def detect_vehicles(frame, model, conf_threshold=0.5):
    # Preprocess the frame
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = model(frame_rgb, stream=True, verbose=False)
    detections = []

    for result in results:
        filtered_boxes = [box for box in result.boxes if box.conf[0] >= conf_threshold]
        result.boxes = filtered_boxes
        detections.append(result)

    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
        gc.collect()

    return detections

## Upload and Process Video

In [ ]:
# Upload video file
uploaded = files.upload()
video_path = list(uploaded.keys())[0]

In [ ]:
def process_video(video_path, conf_threshold=0.5):
    cap = cv2.VideoCapture(video_path)

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    output_path = 'detected_output.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frame_count = 0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1

        detections = detect_vehicles(frame, model, conf_threshold)

        for detection in detections:
            boxes = detection.boxes
            for box in boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = float(box.conf[0])
                cls = int(box.cls[0])
                label = model.names[cls]
                if label in label_colors:
                    color = label_colors[label]
                    label_text = f'{label.capitalize()} {conf:.2f}'
                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                    (text_width, text_height), baseline = cv2.getTextSize(
                        label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
                    cv2.rectangle(frame, (x1, y1 - text_height - baseline - 1),
                                (x1 + text_width, y1), color, thickness=cv2.FILLED)
                    cv2.putText(frame, label_text, (x1, y1 - 4),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

        out.write(frame)

        #print(f'Processed {frame_count}/{total_frames} frames ({(frame_count/total_frames)*100:.1f}%)')
        print(f'\rProcessed {frame_count}/{total_frames} frames ({(frame_count/total_frames)*100:.1f}%)', end='', flush=True)

    cap.release()
    out.release()

    print('\nVideo processing completed!')
    return output_path

In [ ]:
# Process the video
output_path = process_video(video_path, conf_threshold=0.5)

In [ ]:
# Download the processed video
files.download('detected_output.mp4')